# MIT 808 — KDD EDA
## Part 1: Data Loading and Preprocessing
Loading Excel tables, filtering cohorts and preparing variables.

In [7]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
DATA_PATH = Path("41586_2022_5154_MOESM3_ESM.xlsx")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

In [8]:
xl = pd.ExcelFile(DATA_PATH)
s1 = pd.read_excel(xl, "TableS1_clinical_data")
s5a = pd.read_excel(xl, "TableS5_TCGA")
s5b = pd.read_excel(xl, "TableS5_ssm_driver")
s6 = pd.read_excel(xl, "TableS6_integrative_iCluster")
s9 = pd.read_excel(xl, "TableS9_enrichment")
s2 = pd.read_excel(xl, "TableS2_Drivers_hg38")
s11 = pd.read_excel(xl, "TableS11_evolution")

print("S1:", s1.shape, "| S5a:", s5a.shape, "| S5b:", s5b.shape)
print("S6:", s6.shape, "| S9:", s9.shape, "| S2:", s2.shape, "| S11:", s11.shape)

S1: (183, 49) | S5a: (185, 8) | S5b: (184, 651)
S6: (183, 6) | S9: (183, 11) | S2: (183, 7) | S11: (180, 36)


In [9]:
sa = s1[s1["Country"] == "South Africa"].copy()
au = s1[s1["Country"] == "Australia"].copy()

sa_ids = set(sa["Sample_id"].astype(str))
sa_ids_T = {sid + "-T" for sid in sa_ids}
au_ids = set(au["Sample_id"].astype(str))

print("SA:", len(sa), "| AU:", len(au))
display(sa["Genetic_ancestry"].value_counts())

SA: 123 | AU: 53


Genetic_ancestry
African     113
European      5
Admixed       5
Name: count, dtype: int64

In [ ]:
sa["PSA"] = pd.to_numeric(sa["PSA"], errors="coerce")
au["PSA"] = pd.to_numeric(au["PSA"], errors="coerce")

sa["ISUP_GG"] = pd.to_numeric(sa["ISUP_GG"].replace({"1-2": 2}), errors="coerce")
au["ISUP_GG"] = pd.to_numeric(au["ISUP_GG"].replace({"1-2": 2}), errors="coerce")


def age_midpoint(age_str):
    if pd.isna(age_str):
        return np.nan
    try:
        a, b = str(age_str).split("-")
        return (int(a) + int(b)) / 2
    except Exception:
        return np.nan


sa["Age_numeric"] = sa["Age"].apply(age_midpoint)
au["Age_numeric"] = au["Age"].apply(age_midpoint)
sa["PSA_log"] = np.log10(sa["PSA"].replace(0, np.nan))
au["PSA_log"] = np.log10(au["PSA"].replace(0, np.nan))
sa["Risk_class"] = sa["ISUP_GG"].apply(
    lambda x: np.nan if pd.isna(x) else ("High Risk" if x >= 3 else "Low Risk")
)
au["Risk_class"] = au["ISUP_GG"].apply(
    lambda x: np.nan if pd.isna(x) else ("High Risk" if x >= 3 else "Low Risk")
)
sa["Germline_SNVs"] = pd.to_numeric(sa["Germline_SNVs"], errors="coerce")
sa["Germline_indels"] = pd.to_numeric(sa["Germline_indels"], errors="coerce")
sa["chromothripsis_int"] = sa["chromothripsis_positive"].astype(int)
sa["msi_binary"] = (sa["msi"] == "MSI").astype(int)

Prétraitement terminé.
